In [1]:
import os
import numpy as np

In [3]:
from tensorflow.keras.applications import ResNet50

from tensorflow.keras.applications.resnet50 import (
    preprocess_input
)

from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

from tqdm import tqdm

In [10]:
dataset_path=r"sampled_sneakers_1024x1024"

In [5]:
base_model=ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [6]:
x=GlobalAveragePooling2D()(            #without pooling output shape (7x7x2048) -> pooling converts it into 2048 dim vector.
    base_model.output
)

feature_extractor=Model(
    inputs=base_model.input,
    outputs=x
)

In [7]:
def preprocess_image(img_path):
    img=image.load_img(
        img_path,
        target_size=(224,224)
    )

    img_array=image.img_to_array(img)

    img_array=np.expand_dims(
        img_array,
        axis=0
    )

    img_array=preprocess_input(img_array)

    return img_array

In [11]:
image_paths=[]

for file in os.listdir(dataset_path):
    if file.lower().endswith(
        ('.jpg', '.jpeg', '.png')
    ):
        full_path=os.path.join(
            dataset_path,
            file
        )

        image_paths.append(full_path)

print("Total images: ", len(image_paths))

Total images:  2000


In [12]:
features=[]
filenames=[]

for img_path in tqdm(image_paths):        #tqdm just used for showing bar progress in loop.
    try:
        processed_img=preprocess_image(img_path)

        embedding=feature_extractor.predict(
            processed_img,
            verbose=0
        )

        features.append(
            embedding.flatten()
        )

        filenames.append(
            os.path.basename(img_path)
        )

    except Exception as e:
        print(
            f"Error with {img_path}: {e}"            
        )

100%|██████████████████████████████████████████████████████████████████████████████| 2000/2000 [18:58<00:00,  1.76it/s]


In [13]:
features=np.array(features)
print(features.shape)

(2000, 2048)


In [14]:
np.save(
    "shoe_image_embeddings.npy",
    features
)

np.save(
    "shoe_image_filenames.npy",
    filenames
)